# SchizophCohort — define the schizophrenia case cohort

Mirrors the methodology used in `ADCohort.ipynb`, applied to schizophrenia.

**ICD codes** (ICD-10-CM, matches Naomi's `Schizoph.ipynb`):
- F20.0  Paranoid schizophrenia
- F20.1  Disorganized schizophrenia
- F20.2  Catatonic schizophrenia
- F20.3  Undifferentiated schizophrenia
- F20.5  Residual schizophrenia
- F20.81 Schizophreniform disorder
- F20.89 Other schizophrenia
- F20.9  Schizophrenia, unspecified

(F20.4 post-schizophrenic depression, F20.6 simple schizophrenia, and F20.8 other excl. 81/89
are deliberately omitted, following Naomi's original selection.)

**Data source**: `/home/jupyter/2836994-data2` (full adult cohort, ~2.64M persons).
Schizophrenia is early-onset, so the age >=45 subset would severely undercount cases.

**Inclusion criteria** (liberal, matching Naomi's original approach):
1. >=1 condition_occurrence row with `condition_source_value` matching the F20 codes above
2. Age at first code BETWEEN 10 and 30 years
   - Upper bound 30 matches Naomi's `Schizoph.ipynb`, captures the peak onset window for schizophrenia
   - Lower bound 10 is a sanity floor (no plausible schizophrenia in early childhood)

**Filters NOT applied** (deliberately):
- No `condition_type_concept_id` filter (Naomi's notebook used `= 32020`; we keep all sources
  and inspect the distribution as a sanity check)
- No `>=2 codes` requirement; any single coded encounter counts
- No `>=30 day span` requirement
- No sex/race/ethnicity restriction

(If these turn out to be needed after looking at the data, they can be added in a follow-up.)


In [ ]:
## pip3 install pyarrow fastparquet matplotlib seaborn
## Got to Kernel --> Restart Kernel after installing the libraries

import json
import pandas as pd
import os
import copy
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib


In [ ]:
## Configure the path for use in the sql-based query
path = '/home/jupyter/2836994-data2'


In [ ]:
## Schizophrenia cohort query
## Pulls ICD-10 F20.0/.1/.2/.3/.5/.81/.89/.9 (matches Naomi's Schizoph.ipynb selection).
## NOT filtering by condition_type_concept_id here (broad pull, inspect distribution below).
## NOT applying age filter here (will be applied as a cohort-level filter after summary).

query = f"""
SELECT
  per.person_id
  ,per.birth_datetime AS BIRTH_DATE
  ,cond.condition_source_value AS ICD
  ,cond.condition_start_datetime AS START_DATE
  ,cond.condition_type_concept_id AS COND_TYPE
FROM '{path}/person/*.parquet' per
  INNER JOIN '{path}/condition_occurrence/*.parquet' cond
    ON per.person_id = cond.person_id
    AND (
         cond.condition_source_value LIKE 'F20.0%'
      OR cond.condition_source_value LIKE 'F20.1%'
      OR cond.condition_source_value LIKE 'F20.2%'
      OR cond.condition_source_value LIKE 'F20.3%'
      OR cond.condition_source_value LIKE 'F20.5%'
      OR cond.condition_source_value LIKE 'F20.81%'
      OR cond.condition_source_value LIKE 'F20.89%'
      OR cond.condition_source_value LIKE 'F20.9%'
    )
"""


In [ ]:
## Run query, materialize to pandas
df_sch_raw = duckdb.sql(query).to_df()
df_sch_raw.shape


In [ ]:
## Quick sanity checks before deciding the cohort definition

# 1. ICD code distribution -- which subtypes show up, and at what counts
df_sch_raw['ICD'].value_counts()


In [ ]:
# 2. condition_type_concept_id distribution
# (32020 = EHR encounter dx, 32840 = EHR problem list, 32817 = EHR claim, etc.)
df_sch_raw['COND_TYPE'].value_counts()


In [ ]:
# 3. Per-person event counts
events_per_person = df_sch_raw.groupby('person_id').size()
print(f"Unique persons with any Schizophrenia code: {df_sch_raw['person_id'].nunique()}")
print(f"Median Schizophrenia codes per person: {events_per_person.median()}")
print(f"Persons with >=2 codes: {(events_per_person >= 2).sum()}")


In [ ]:
# 4. Age at first Schizophrenia code (peak should be 16-30, tail extending older)
df_first = (df_sch_raw
            .sort_values(['person_id', 'START_DATE'])
            .groupby('person_id')
            .first()
            .reset_index())
df_first['AGE_AT_FIRST_SCH'] = (
    (pd.to_datetime(df_first['START_DATE']) - pd.to_datetime(df_first['BIRTH_DATE']))
    .dt.days / 365.25
)
df_first['AGE_AT_FIRST_SCH'].describe()


In [ ]:
# 5. Visualize age distribution
plt.figure(figsize=(8, 4))
sns.histplot(df_first['AGE_AT_FIRST_SCH'].dropna(), bins=60)
plt.xlabel('Age at first Schizophrenia code (years)')
plt.ylabel('Count')
plt.title(f'Schizophrenia raw: age at first diagnosis (n={len(df_first)})')
plt.axvline(30, color='red', linestyle='--', label='30 (upper cutoff for case definition)')
plt.axvline(10, color='orange', linestyle='--', label='10 (lower sanity floor)')
plt.legend()
plt.show()


In [ ]:
# 6. Investigate the implausibly young cases (<10) and late cases (>30)
implausibly_young = df_first[df_first['AGE_AT_FIRST_SCH'] < 10]
late_onset = df_first[df_first['AGE_AT_FIRST_SCH'] > 30]
print(f'Persons with first Schizophrenia code before age 10: {len(implausibly_young)}')
print(f'Persons with first Schizophrenia code after age 30:  {len(late_onset)}')

if len(implausibly_young) > 0:
    print('\nSample of pre-10 cases (likely coding errors):')
    display(implausibly_young[['person_id', 'BIRTH_DATE', 'START_DATE', 'ICD', 'AGE_AT_FIRST_SCH']].head(10))


In [ ]:
## Apply cohort filter: age at first code BETWEEN 10 and 30
## No >=2 code requirement; no >=30 day span requirement -- single liberal cutoff on age only.

MIN_AGE = 10
MAX_AGE = 30

df_sch_raw['START_DATE'] = pd.to_datetime(df_sch_raw['START_DATE'])
df_sch_raw['BIRTH_DATE'] = pd.to_datetime(df_sch_raw['BIRTH_DATE'])

g = df_sch_raw.groupby('person_id').agg(
    first_date=('START_DATE', 'min'),
    last_date=('START_DATE', 'max'),
    n_unique_days=('START_DATE', 'nunique'),
    n_total_codes=('ICD', 'count'),
    birth_date=('BIRTH_DATE', 'first'),
)
g['age_at_first'] = (g['first_date'] - g['birth_date']).dt.days / 365.25
g['span_days'] = (g['last_date'] - g['first_date']).dt.days

cohort = g[
    (g['age_at_first'] >= MIN_AGE) &
    (g['age_at_first'] <= MAX_AGE)
].index.tolist()

print(f"Final Schizophrenia cohort (age {MIN_AGE}-{MAX_AGE}, any single code OK): {len(cohort)}")
print(f"  Excluded by age <{MIN_AGE}: {(g['age_at_first'] < MIN_AGE).sum()}")
print(f"  Excluded by age >{MAX_AGE}: {(g['age_at_first'] > MAX_AGE).sum()}")
print()
print(f"For reference (NOT applied here):")
print(f"  Of {len(cohort)} retained, persons with only 1 code on 1 day: "
      f"{((g.loc[cohort, 'n_unique_days'] == 1)).sum()}")
print(f"  Of {len(cohort)} retained, persons with >=2 codes and >=30 day span: "
      f"{((g.loc[cohort, 'n_unique_days'] >= 2) & (g.loc[cohort, 'span_days'] >= 30)).sum()}")


In [ ]:
## Build the final cohort table: one row per person, INDEX_DATE = first Schizophrenia code

df_cohort = (df_sch_raw[df_sch_raw['person_id'].isin(cohort)]
             .sort_values(['person_id', 'START_DATE'])
             .groupby('person_id')
             .agg(
                 BIRTH_DATE=('BIRTH_DATE', 'first'),
                 INDEX_DATE=('START_DATE', 'min'),       # first Schizophrenia code = index date
                 LAST_SCH_DATE=('START_DATE', 'max'),
                 N_SCH_CODES=('ICD', 'count'),
                 N_SCH_DAYS=('START_DATE', 'nunique'),
                 FIRST_SCH_ICD=('ICD', 'first'),
                 EVER_F20_0=('ICD', lambda x: (x.str.startswith('F20.0')).any()),
                 EVER_F20_1=('ICD', lambda x: (x.str.startswith('F20.1')).any()),
                 EVER_F20_2=('ICD', lambda x: (x.str.startswith('F20.2')).any()),
                 EVER_F20_3=('ICD', lambda x: (x.str.startswith('F20.3')).any()),
                 EVER_F20_5=('ICD', lambda x: (x.str.startswith('F20.5')).any()),
                 EVER_F20_81=('ICD', lambda x: (x.str.startswith('F20.81')).any()),
                 EVER_F20_89=('ICD', lambda x: (x.str.startswith('F20.89')).any()),
                 EVER_F20_9=('ICD', lambda x: (x.str.startswith('F20.9')).any()),
             )
             .reset_index())

df_cohort['AGE_AT_INDEX'] = (
    (df_cohort['INDEX_DATE'] - df_cohort['BIRTH_DATE']).dt.days / 365.25
)

print(df_cohort.shape)
df_cohort.head()


In [ ]:
## Save cohort
out_dir = '/Users/sw2572/Documents/codes/gitcodes/ynhhRunning/outputs'
os.makedirs(out_dir, exist_ok=True)
df_cohort.to_parquet(f'{out_dir}/sch_cohort.parquet', index=False)
print(f"Saved {len(df_cohort)} Schizophrenia patients to {out_dir}/sch_cohort.parquet")
